# MV00 B16 — Epoch 95 to 100, governed AMP128 numerical-recovery branch

**Important: This is a checkpoint-producing training continuation, not another diagnostic, and not a bitwise-exact continuation of the original AMP trajectory.** The original frozen source commit is preserved and the validated Epoch-95 checkpoint remains immutable, but after a clean two-rank accumulation boundary the restored AMP scale is changed from 1024 to fixed 128. The safety guards, model, data, hyperparameters, G4 per-epoch capture, and terminal B-VAL remain intact.

## Why this branch exists
At original scale 1024, Epoch-96 batch index 696 caused 1024/4096 NaN/Inf elements in rank0's `gradient.model.23.cv2.1.2.weight`, terminating DDP; R3 re-executed this batch with scale128 successfully on both ranks. The purpose of this governed continuation is to **finish Epochs 96–100 while reporting the altered numerical policy honestly**. No result is automatically promoted into the original frozen policy.

## Kaggle settings
Use 2×Tesla T4, Internet OFF, and the same seven scientific datasets, frozen DDP Git bundle, and two Epoch-95 recovery inputs (`mv00-epoch95` and `mv00-epoch95-best-recovery`) that passed the previous 11 preflight gates. Select **Save Version → Save & Run All** in a clean Kaggle notebook; preserve output. No B-TEST access. The immutable source is `f440d329d4544ec08ad63c1972bcf9a8b8a7eed8`; starting G4 SHA `b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745`, 21,767 optimizer updates, Epoch 95 complete.

## Training and checkpoint governance
At both ranks' first clean gradient-accumulation boundary (known R3 batch index 1), apply fixed loss scale128. Each native optimizer step retains all original fail-closed checks and subsequently calls GradScaler.update(new_scale=128), keeping scale fixed. The trained numerical trajectory is intentionally amended. At each completed nonterminal epoch, full 2-rank G4 is published and SHA verified, with an `AMP128_RECOVERY_POLICY.json` file and policy SHA embedded in `ACCEPTANCE.json`. If the session ends at a completed epoch, preserve the newest G4 and attach it for the next continuation (do not revert to Epoch 95 or silently combine trajectories). After Epoch 100, require exact 3,049 B-VAL image count, finite metrics, and SHA-pinned `terminal/TERMINAL.json`, `best.pt`, and `last.pt`. The terminal is a candidate pending independent off-session verification, not a validated scientific comparison.

## What to upload here after Kaggle
Share the final `MV00_CONTINUOUS_LOG.txt` plus `terminal/TERMINAL.json`, `AMP128_RECOVERY_POLICY.json` and the terminal files' SHA verification output. If Kaggle stops before Epoch100, share `g4/ACCEPTANCE.json` and `MV00_CONTINUOUS_LOG.txt`, and retain the entire G4 package (including both rank checkpoint `.pt` files) for the next session. Never attach or evaluate B-TEST before authorization.


In [ ]:
# Exact Epoch-95 MV00 G4 seven-file staging; no training or B-TEST.
from pathlib import Path
import zipfile, json, hashlib, shutil, stat

RECOVERY_PIN="b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745"
RECOVERY_UPDATES=21767
RECOVERY_ZIP_NAME="MV00_EPOCH95_RECOVERY_COMPLETE.zip"
RECOVERY_PREFIX="MV00_B16_CONTINUOUS_OUTPUT/g4/"
RECOVERY_ROOT=Path("/kaggle/working/MV00_E95_INPUT_STAGED")
RECOVERY_G4=RECOVERY_ROOT/"MV00_B16_CONTINUOUS_OUTPUT"/"g4"
RECOVERY_FILES={
 "best.pt": "3e737d0318eae803a638de0bd28365499b87a377ba43de8a4b1d251eee73605e",
 "MV00_B16_FRESH_ACCEPTANCE_epoch095_rank0.pt": "4578d1c1b02124ec69f185f875bc78856821814b48c8e0b1b5ce42d6acc8d1a1",
 "MV00_B16_FRESH_ACCEPTANCE_epoch095_rank0.receipt.json": "508faf76a52ee81978b862dbeab2dd43a5e35f0779c5c94fd3b415861b0c7dc5",
 "MV00_B16_FRESH_ACCEPTANCE_epoch095_rank1.pt": "117c7762391951ca5ceaafeecb905ea3049122625ce177ca697e60ebbcb0da17",
 "MV00_B16_FRESH_ACCEPTANCE_epoch095_rank1.receipt.json": "f7025210b581a8dc4aa262e1f499c6231cd9c34770b59a813498dbbe2461ef4d",
 "MV00_B16_FRESH_ACCEPTANCE_epoch095_two_rank_manifest.json": "b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745"
}

def r_need(ok,reason):
    if not ok:
        raise RuntimeError("MV00_E95_"+reason)

def r_sha(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for part in iter(lambda:f.read(4*1024*1024),b""):
            h.update(part)
    return h.hexdigest()

r_need(not RECOVERY_ROOT.exists(),"STAGING_NOT_CLEAN")
attached=Path("/kaggle/input")
r_need(attached.is_dir(),"NO_ATTACHED_DATASETS")
archives=sorted(x for x in attached.rglob(RECOVERY_ZIP_NAME)
                if x.is_file() and not x.is_symlink())
r_need(len(archives)<=1,"DUPLICATE_ZIPS")
if archives:
    print("PINNED_E95_RECOVERY_ARCHIVE:",archives[0],flush=True)
    with zipfile.ZipFile(archives[0]) as z:
        members=set(z.namelist())
        required={RECOVERY_PREFIX+n for n in ["ACCEPTANCE.json",*RECOVERY_FILES]}
        r_need(required.issubset(members),"ZIP_MISSING_FILES")
        RECOVERY_G4.mkdir(parents=True,exist_ok=False)
        for key in sorted(required):
            info=z.getinfo(key)
            r_need(not info.is_dir() and
                not stat.S_ISLNK(info.external_attr>>16),"ZIP_UNSAFE_FILE")
            dest=RECOVERY_G4/Path(key).name
            with z.open(info) as input_file, dest.open("xb") as output_file:
                shutil.copyfileobj(input_file,output_file,4*1024*1024)
else:
    # Supports Kaggle datasets that auto-unpack the ZIP before mounting.
    found=[]
    for candidate in attached.rglob("ACCEPTANCE.json"):
        if candidate.is_file() and not candidate.is_symlink() and candidate.parent.name=="g4":
            try:
                a=json.loads(candidate.read_text("utf-8"))
                if a.get("manifest_sha256")==RECOVERY_PIN and a.get("completed_epoch")==95:
                    found.append(candidate)
            except (OSError,ValueError):
                pass
    r_need(len(found)==1,"EXTRACTED_E95_NOT_UNIQUE")
    RECOVERY_G4.mkdir(parents=True,exist_ok=False)
    for name in ["ACCEPTANCE.json",*RECOVERY_FILES]:
        source=found[0].parent/name
        if name=="best.pt" and not source.is_file():
            # The original portable recovery omitted best.pt. Accept an
            # independently mounted file ONLY with the exact pinned digest.
            best_candidates=sorted(
                item for item in attached.rglob("best.pt")
                if item.is_file() and not item.is_symlink()
                and r_sha(item)==RECOVERY_FILES["best.pt"]
            )
            r_need(bool(best_candidates),"MISSING_EXACT_SHA_PINNED_BEST_PT")
            source=best_candidates[0]
            print("MV00_E95_INDEPENDENT_BEST_PT:",source,flush=True)
        r_need(source.is_file() and not source.is_symlink(),"MISSING_"+name)
        shutil.copyfile(source,RECOVERY_G4/name)

receipt=json.loads((RECOVERY_G4/"ACCEPTANCE.json").read_text("utf-8"))
r_need(receipt.get("schema")=="MV00_DDP_B16_CONTINUOUS_OUTPUT_V1","RECEIPT_SCHEMA")
r_need(receipt.get("status")=="MV00_RESUMABLE_EPOCH_G4_CANDIDATE","RECEIPT_STATUS")
r_need(receipt.get("run_id")=="MV00_B16_FRESH_ACCEPTANCE","RUN_ID")
r_need(receipt.get("source_release_sha")=="f440d329d4544ec08ad63c1972bcf9a8b8a7eed8","FROZEN_SOURCE")
r_need(receipt.get("b_test_access")=="NONE","BTEST")
r_need(receipt.get("completed_epoch")==95 and receipt.get("next_epoch")==95,"EPOCH")
r_need(receipt.get("optimizer_updates")==RECOVERY_UPDATES,"OPTIMIZER_UPDATES")
r_need(receipt.get("manifest_sha256")==RECOVERY_PIN,"MANIFEST_SHA")
r_need(receipt.get("files_sha256")==RECOVERY_FILES,"SIX_MEMBER_PIN")
r_need(receipt.get("selected_best_sha256")==RECOVERY_FILES["best.pt"],"BEST_PIN")
for name,digest in RECOVERY_FILES.items():
    target=RECOVERY_G4/name
    r_need(target.is_file() and not target.is_symlink(),"MISSING_"+name)
    r_need(r_sha(target)==digest,"SHA_"+name)
manifest=json.loads((RECOVERY_G4/"MV00_B16_FRESH_ACCEPTANCE_epoch095_two_rank_manifest.json").read_text("utf-8"))
c=manifest["cursor"]
r_need(manifest["complete"] is True and manifest["world_size"]==2,"MANIFEST")
r_need(manifest["identity"]==receipt["identity"] and manifest["identity"]["arm"]=="MV00","IDENTITY")
r_need(manifest["b_test_access"]=="NONE","MANIFEST_BTEST")
r_need(c["completed_epoch"]==94 and c["next_epoch"]==95 and
       c["sampler_next_epoch"]==95 and c["optimizer_updates"]==RECOVERY_UPDATES and
       c["steps_per_rank"]==890 and c["global_batch"]==16,"CURSOR")
print("MV00 EPOCH-95 SEVEN-FILE RECOVERY: PASS",flush=True)
print("EXACT MANIFEST SHA256:",RECOVERY_PIN,flush=True)
print("OPTIMIZER UPDATES:",RECOVERY_UPDATES,flush=True)
print("NEXT EPOCH: 96/100; B-TEST: SEALED",flush=True)


In [ ]:
# ============================================================
# MV00 EPOCH-96 READ-ONLY CHECKPOINT PREFLIGHT
# ZERO TRAINING / NO MODEL LOAD / B-TEST SEALED
# ============================================================
from pathlib import Path
import hashlib, json, re
import torch

PIN = "b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745"
SOURCE = "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"
RUN = "MV00_B16_FRESH_ACCEPTANCE"
INPUT = Path("/kaggle/working/MV00_E95_INPUT_STAGED")
BUNDLE = Path("/kaggle/input/datasets/awaishit/resema-ddp-b16-acceptance/RESEMA_DDP_B16_ACCEPTANCE.bundle")
BUNDLE_PIN = "4dc26b7277d58ed0b1fbcc1f4fac68aaf7ad417951e735bc8be2b040764861bb"
DATASETS = {
 "n1": "/kaggle/input/datasets/awaishit/detection-frozen-git-n1-accepted-r1",
 "handoff": "/kaggle/input/datasets/awaishit/detection-project-training-handoff-r1",
 "legacy": "/kaggle/input/datasets/awaishit/detection-legacy-notebook-files-flat-r1",
 "split_b": "/kaggle/input/datasets/utopianstar/grazpedwri-dx-split-b",
 "assets": "/kaggle/input/datasets/nithoukhan/d4e-r12-accepted-assets",
 "archives": "/kaggle/input/datasets/nithoukhan/d4e-r12-raw-archives-v1",
 "evidence": "/kaggle/input/datasets/nithoukhan2/d4f-r7d-legacy-exact-source-evidence-r1",
}

def need(ok,code):
 if not ok:
  raise RuntimeError("MV00_EPOCH96_PREFLIGHT_" + code)

def sha256(p):
 h = hashlib.sha256()
 with Path(p).open("rb") as f:
  for chunk in iter(lambda:f.read(4*1024*1024), b""):
   h.update(chunk)
 return h.hexdigest()

def read_json(p):
 return json.loads(Path(p).read_text("utf-8"))

print("="*68)
print("MV00 EPOCH-96 READ-ONLY PRE-TRAINING VERIFICATION")
print("="*68)

# 1. GPU topology and all independent project inputs
need(torch.cuda.is_available() and torch.cuda.device_count() == 2, "TWO_CUDA_GPUS")
for rank in (0,1):
 name = torch.cuda.get_device_name(rank)
 need("T4" in name.upper(), "GPU_MUST_BE_T4")
 print("GPU", rank, name)
print("GATE 1 — TWO T4 GPUs: PASS")

for label,location in DATASETS.items():
 p=Path(location)
 need(p.is_dir() and not p.is_symlink(), "DATASET_"+label.upper())
print("GATE 2 — ALL SEVEN SCIENTIFIC DATASETS: PASS")
need(BUNDLE.is_file() and not BUNDLE.is_symlink() and
     sha256(BUNDLE)==BUNDLE_PIN, "FROZEN_BUNDLE_HASH")
print("GATE 3 — PINNED FROZEN SOURCE BUNDLE: PASS")

# 2. Search attached output; not allowed to substitute an old G4
roots=[]
if INPUT.is_dir():
 roots.append(INPUT)
else:
 print("EXPECTED_INPUT_PATH_NOT_FOUND:",INPUT)
 notebook_root=Path("/kaggle/input/notebooks")
 if notebook_root.is_dir():
  roots.append(notebook_root)
 dataset_root=Path("/kaggle/input/datasets")
 if dataset_root.is_dir():
  for owner in sorted(dataset_root.iterdir()):
   if not owner.is_dir() or owner.is_symlink():
    continue
   for d in sorted(owner.iterdir()):
    if d.is_dir() and not d.is_symlink() and any(
      name in d.name.lower() for name in ("mv00","ddp-b16","continuous","grazpedwri")
    ):
     roots.append(d)
roots=list(dict.fromkeys(roots))
matches=[]
seen=[]
for root in roots:
 for receipt_path in sorted(root.rglob("ACCEPTANCE.json")):
  if (not receipt_path.is_file() or receipt_path.is_symlink() or
      receipt_path.parent.name!="g4"):
   continue
  try:
   metadata=read_json(receipt_path)
  except (ValueError,OSError):
   continue
  seen.append((str(receipt_path),metadata.get("manifest_sha256")))
  if (metadata.get("manifest_sha256")==PIN and
      metadata.get("run_id")==RUN and
      metadata.get("source_release_sha")==SOURCE and
      metadata.get("completed_epoch")==95):
   matches.append(receipt_path)
matches=list(dict.fromkeys(matches))
if len(matches)!=1:
 print("VISIBLE_G4_RECEIPTS:",seen)
need(len(matches)==1,"PINNED_EPOCH95_G4_MISSING_OR_AMBIGUOUS")
g4=matches[0].parent
receipt=read_json(matches[0])
print("SELECTED G4:",g4)
print("GATE 4 — EXACT EPOCH-95 CHECKPOINT FOUND: PASS")

# 3. Read verified receipt and hash every member
need(receipt.get("schema")=="MV00_DDP_B16_CONTINUOUS_OUTPUT_V1",
     "RECEIPT_SCHEMA")
need(receipt.get("status")=="MV00_RESUMABLE_EPOCH_G4_CANDIDATE",
     "RECEIPT_STATUS")
need(receipt.get("completed_epoch")==95 and
     receipt.get("next_epoch")==95 and
     receipt.get("b_test_access")=="NONE","RECEIPT_CURSOR_POLICY")
updates=receipt.get("optimizer_updates")
need(type(updates) is int and updates>20432,"UPDATE_CURSOR")
print("RESTORE OPTIMIZER UPDATES:",updates)
print("GATE 5 — CHECKPOINT RECEIPT AND CURSOR: PASS")

files=receipt.get("files_sha256")
need(type(files) is dict and len(files)==6 and "best.pt" in files,
     "SIX_MEMBER_G4")
for name,expected in sorted(files.items()):
 need(Path(name).name==name and
      isinstance(expected,str) and
      re.fullmatch(r"[0-9a-f]{64}",expected) is not None,
      "UNSAFE_G4_MEMBER")
 p=g4/name
 need(p.is_file() and not p.is_symlink() and sha256(p)==expected,
      "MEMBER_SHA_"+name)
 print("SHA256 PASS:",name)
print("GATE 6 — ALL SIX SAVED G4 MEMBERS: PASS")

# 4. Check exact pinned manifest and every rank receipt
manifest_names=[x for x in files if x.endswith("_two_rank_manifest.json")]
need(len(manifest_names)==1,"MANIFEST_NAME")
manifest_path=g4/manifest_names[0]
need(sha256(manifest_path)==PIN,"EXTERNAL_MANIFEST_PIN_MISMATCH")
manifest=read_json(manifest_path)
identity,cursor=manifest["identity"],manifest["cursor"]
need(manifest.get("schema")=="VGRA_DDP_B16_G4_EPOCH_PAIR_CANDIDATE_V1" and
     manifest.get("complete") is True and
     manifest.get("world_size")==2 and
     set(manifest.get("ranks",{}))=={"0","1"} and
     manifest.get("b_test_access")=="NONE",
     "MANIFEST_SCHEMA_OR_POLICY")
print("GATE 7 — INDEPENDENT MANIFEST SHA256: PASS")

need(identity["arm"]=="MV00" and
     identity["run_id"]==RUN and
     identity["ddp_release_sha"]==SOURCE,
     "FROZEN_SCIENTIFIC_IDENTITY")
need(files["best.pt"]==receipt["selected_best_sha256"],"BEST_SHA")
print("GATE 8 — MV00 SCIENTIFIC IDENTITY: PASS")

need(cursor["completed_epoch"]==94 and
     cursor["next_epoch"]==95 and
     cursor["sampler_next_epoch"]==95 and
     cursor["steps_per_rank"]==890 and
     cursor["global_batch"]==16 and
     cursor["optimizer_updates"]==updates and
     cursor["accumulate"]>=1,
     "EPOCH96_OPTIMIZER_OR_SAMPLER_CURSOR")
print("GATE 9 — EPOCH-96 RESUME CURSOR: PASS")

for rank in (0,1):
 member=manifest["ranks"][str(rank)]
 need(files[member["filename"]]==member["sha256"],"RANK_PT_SHA")
 need(files[member["receipt_filename"]]==member["receipt_sha256"],
      "RANK_RECEIPT_SHA")
 rr=read_json(g4/member["receipt_filename"])
 need(rr["rank"]==rank and rr["identity"]==identity and
      rr["cursor"]==cursor and rr["sha256"]==member["sha256"],
      "RANK_RECEIPT_IDENTITY")
 print("RANK",rank,"PASS")
print("GATE 10 — BOTH RANK RECEIPTS: PASS")

for path in (
 "/kaggle/working/MV00_B16_CONTINUOUS",
 "/kaggle/working/MV00_B16_CONTINUOUS_OUTPUT",
):
 need(not Path(path).exists(),"WORK_NAMESPACE_NOT_CLEAN")
print("GATE 11 — CLEAN TRAINING WORKSPACE: PASS")

print("="*68)
print("MV00 EPOCH-96 READ-ONLY PRE-TRAINING VERIFICATION: PASS")
print("COMPLETED EPOCHS: 95")
print("NEXT TRAINING EPOCH: 96")
print("RESTORE OPTIMIZER UPDATES:",updates)
print("MANIFEST:",PIN)
print("TRAINING EXECUTED: NO")
print("B-TEST SEALED")
print("="*68)


In [ ]:
WORKER_CODE = "\"\"\"Pinned-source MV00 DDP-B16 multi-epoch Kaggle continuation (nonofficial).\n\nThe scientific source stays at f440d329d4544ec08ad63c1972bcf9a8b8a7eed8.\nReal torchrun ranks separately authenticate R0, source stage and external G4;\nthe fixed native trainer handles all optimizer, augmentation and B-VAL math.\nOnly the verified epoch-G4 boundary may stop a session. B-TEST is sealed.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport sys\nimport time\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\n\nROOT = Path(\"/kaggle/working/MV00_B16_CONTINUOUS\")\nREPO = ROOT / \"repo\"\nG4 = ROOT / \"g4\"\nOUT = Path(\"/kaggle/working/MV00_B16_CONTINUOUS_OUTPUT\")\nPLAN = ROOT / \"plan.json\"\nEXTERNAL_PIN = os.environ[\"DDP_B16_PREVIOUS_MANIFEST_SHA256\"]\nPOLICY_ID = \"MV00_AMP128_CAPPED_FAILCLOSED_RECOVERY_V1\"\nPOLICY_SHA = os.environ[\"DDP_B16_AMP128_POLICY_SHA256\"]\nBUDGET_HOURS = float(os.environ.get(\"DDP_B16_SESSION_HOURS\", \"9.0\"))\nsys.dont_write_bytecode = True\nsys.path.insert(0, str(REPO))\n\nfrom research.r9_training.ddp_b16.ddp_worker import review_worker_inputs, _unique_json\nfrom research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission\nfrom research.r9_training.ddp_b16.ddp_rank_admission import require_review_consensus\nfrom research.r9_training.ddp_b16.ddp_native_binding import make_native_binding\nfrom research.r9_training.ddp_b16.ddp_native_binding_consensus import require_native_binding_consensus\nfrom research.r9_training.ddp_b16.ddp_native_bootstrap_candidate import verify_two_t4_runtime, load_pinned_native_class\nfrom research.r9_training.ddp_b16.ddp_native_state import SCHEMA as STATE_SCHEMA\nfrom research.r9_training.ddp_b16 import ddp_native_g4_hook as hook\n\n\nclass EpochBoundaryStop(Exception):\n    pass\n\n\ndef require(value, code):\n    if not value:\n        raise RuntimeError(\"MV00_MULTI_EPOCH_\" + code)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\"rb\") as f:\n        for part in iter(lambda: f.read(4 * 1024 * 1024), b\"\"):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write_exclusive_json(path, obj):\n    with Path(path).open(\"x\", encoding=\"utf-8\") as f:\n        json.dump(obj, f, sort_keys=True, indent=2, allow_nan=False)\n        f.write(\"\\n\")\n\n\ndef persist_epoch(trainer, result, identity, epoch, previous_sha):\n    \"\"\"Rank 0: SHA-verify all six members and rotate one resumable output.\"\"\"\n    step = epoch + 1\n    filename = f'{identity[\"run_id\"]}_epoch{step:03d}_two_rank_manifest.json'\n    manifest_path = G4 / filename\n    require(manifest_path.is_file() and\n            sha256(manifest_path) == result[\"manifest_sha256\"], \"MANIFEST_HASH\")\n    manifest = json.loads(manifest_path.read_text(\"utf-8\"))\n    require(manifest.get(\"complete\") is True and\n            manifest.get(\"identity\") == identity and\n            manifest.get(\"cursor\", {}).get(\"next_epoch\") == step and\n            manifest[\"cursor\"][\"completed_epoch\"] == epoch and\n            manifest.get(\"b_test_access\") == \"NONE\" and\n            set(manifest[\"ranks\"]) == {\"0\", \"1\"}, \"MANIFEST_CONTENT\")\n    files = {filename: result[\"manifest_sha256\"],\n             \"best.pt\": result[\"selected_best_sha256\"]}\n    for rank in (0, 1):\n        member = manifest[\"ranks\"][str(rank)]\n        files[member[\"filename\"]] = member[\"sha256\"]\n        files[member[\"receipt_filename\"]] = member[\"receipt_sha256\"]\n    require(len(files) == 6, \"SIX_G4_MEMBERS\")\n    temporary = OUT / f\".next_{step:03d}\"\n    require(not temporary.exists(), \"TEMP_EXISTS\")\n    temporary.mkdir()\n    for name, digest in files.items():\n        require(Path(name).name == name, \"PATH_TRAVERSAL\")\n        origin = Path(trainer.best) if name == \"best.pt\" else G4 / name\n        require(origin.is_file() and not origin.is_symlink() and\n                sha256(origin) == digest, \"G4_SOURCE_SHA\")\n        shutil.copyfile(origin, temporary / name)\n        require(sha256(temporary / name) == digest, \"G4_EXPORT_SHA\")\n    require(float(trainer.scaler.get_scale()) == 128.0,\n            \"EPOCH_AMP_SCALE_NOT_128\")\n    require(sha256(OUT / \"AMP128_RECOVERY_POLICY.json\") == POLICY_SHA,\n            \"POLICY_FILE_CHANGED\")\n    shutil.copyfile(OUT / \"AMP128_RECOVERY_POLICY.json\",\n                    temporary / \"AMP128_RECOVERY_POLICY.json\")\n    require(sha256(temporary / \"AMP128_RECOVERY_POLICY.json\") == POLICY_SHA,\n            \"EPOCH_POLICY_SHA\")\n    receipt = {\n        \"schema\": \"MV00_DDP_B16_CONTINUOUS_OUTPUT_V1\",\n        \"status\": \"MV00_RESUMABLE_EPOCH_G4_CANDIDATE\",\n        \"identity\": identity, \"run_id\": identity[\"run_id\"],\n        \"source_release_sha\": identity[\"ddp_release_sha\"],\n        \"completed_epoch\": step, \"next_epoch\": step,\n        \"optimizer_updates\": manifest[\"cursor\"][\"optimizer_updates\"],\n        \"manifest_sha256\": result[\"manifest_sha256\"],\n        \"previous_manifest_sha256\": previous_sha,\n        \"selected_best_sha256\": result[\"selected_best_sha256\"],\n        \"files_sha256\": files, \"b_test_access\": \"NONE\",\n        \"official_training_authorized\": False,\n        \"offsession_restoration_verified\": False,\n        \"amp128_recovery_branch\": True,\n        \"amp128_policy_id\": POLICY_ID,\n        \"amp128_policy_sha256\": POLICY_SHA,\n        \"amp_scale_at_checkpoint\": 128.0,\n    }\n    write_exclusive_json(temporary / \"ACCEPTANCE.json\", receipt)\n    latest, previous = OUT / \"g4\", OUT / \".previous_g4\"\n    require(not previous.exists(), \"PREVIOUS_ROTATION_UNFINISHED\")\n    if latest.exists():\n        latest.rename(previous)\n    temporary.rename(latest)\n    if previous.exists():\n        shutil.rmtree(previous)\n    # Prune only generated work-state copies; the newest exported G4 is intact.\n    for path in G4.iterdir():\n        if path.is_file() and path.name not in files and path.name != \"best.pt\":\n            path.unlink()\n    return receipt\n\n\ndef persist_terminal(trainer, identity):\n    \"\"\"Executed only after native selected-BEST final B-VAL succeeds.\"\"\"\n    require(trainer.epoch == 99 and\n            trainer.validator.seen == 3049 and\n            trainer.validator._d4_fixed_seen == 3049, \"TERMINAL_BVAL\")\n    metrics = {str(k): float(v) for k, v in trainer.metrics.items()}\n    require(bool(metrics) and all(math.isfinite(x) for x in metrics.values()),\n            \"TERMINAL_METRICS\")\n    terminal = OUT / \"terminal\"\n    require(not terminal.exists(), \"TERMINAL_OUTPUT_EXISTS\")\n    terminal.mkdir()\n    files = {}\n    for name, path in ((\"best.pt\", Path(trainer.best)),\n                       (\"last.pt\", Path(trainer.last))):\n        require(path.is_file() and not path.is_symlink(),\n                \"TERMINAL_CHECKPOINT\")\n        files[name] = sha256(path)\n        shutil.copyfile(path, terminal / name)\n        require(sha256(terminal / name) == files[name], \"TERMINAL_SHA\")\n    require(float(trainer.scaler.get_scale()) == 128.0,\n            \"TERMINAL_AMP_SCALE_DRIFT\")\n    shutil.copyfile(OUT / \"AMP128_RECOVERY_POLICY.json\",\n                    terminal / \"AMP128_RECOVERY_POLICY.json\")\n    require(sha256(terminal / \"AMP128_RECOVERY_POLICY.json\") == POLICY_SHA,\n            \"TERMINAL_POLICY_SHA\")\n    receipt = {\n        \"schema\": \"MV00_DDP_B16_TERMINAL_V1\",\n        \"status\": \"EPOCH100_FINISHED_SELECTED_BEST_BVAL_CANDIDATE\",\n        \"identity\": identity, \"completed_epochs\": 100,\n        \"metrics\": metrics, \"operational_val_images\": 3049,\n        \"files_sha256\": files, \"b_test_access\": \"NONE\",\n        \"official_training_authorized\": False,\n        \"amp128_recovery_branch\": True,\n        \"amp128_policy_id\": POLICY_ID,\n        \"amp128_policy_sha256\": POLICY_SHA,\n        \"amp_scale_at_terminal\": 128.0,\n    }\n    write_exclusive_json(terminal / \"TERMINAL.json\", receipt)\n    return receipt\n\n\ndef main():\n    rank, local, world = (int(os.environ[k])\n                          for k in (\"RANK\", \"LOCAL_RANK\", \"WORLD_SIZE\"))\n    require(os.environ.get(\"CUDA_VISIBLE_DEVICES\") == \"0,1\" and\n            (world, rank, local) in ((2, 0, 0), (2, 1, 1)), \"RANKS\")\n    require(0.5 <= BUDGET_HOURS <= 10.5, \"SESSION_BUDGET\")\n    plan = _unique_json(PLAN)\n    policy_file = OUT / \"AMP128_RECOVERY_POLICY.json\"\n    require(policy_file.is_file() and sha256(policy_file) == POLICY_SHA,\n            \"RECOVERY_POLICY_SHA_MISMATCH\")\n    policy = json.loads(policy_file.read_text(\"utf-8\"))\n    require(policy.get(\"policy_id\") == POLICY_ID and\n            policy.get(\"starting_manifest_sha256\") ==\n                \"b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745\" and\n            policy.get(\"amp_scale\") == 128 and\n            policy.get(\"original_restored_amp_scale\") == 1024 and\n            policy.get(\"b_test_access\") == \"NONE\" and\n            policy.get(\"scientific_original_trajectory\") is False,\n            \"RECOVERY_POLICY_CONTENT\")\n    manifests = list(G4.glob(\"*_two_rank_manifest.json\"))\n    require(len(manifests) == 1 and\n            sha256(manifests[0]) == EXTERNAL_PIN, \"EXTERNAL_MANIFEST_PIN\")\n    manifest = json.loads(manifests[0].read_text(\"utf-8\"))\n    identity, cursor = manifest[\"identity\"], manifest[\"cursor\"]\n    require(identity[\"arm\"] == \"MV00\" and\n            identity[\"run_id\"] == plan[\"run_id\"] and\n            identity[\"ddp_release_sha\"] == plan[\"expected_git_head\"] and\n            1 <= cursor[\"next_epoch\"] < 100, \"RESUME_IDENTITY\")\n    require(G4.is_dir() and OUT.is_dir() and not (OUT / \"g4\").exists(),\n            \"NEW_OUTPUT_REQUIRED\")\n\n    dist.init_process_group(\"gloo\", init_method=\"env://\",\n                            rank=rank, world_size=2,\n                            timeout=timedelta(seconds=600))\n    try:\n        review = review_worker_inputs(plan, stage_source=False)\n        admission = verify_staged_admission(plan, review)\n        require_review_consensus(admission)\n        config = _unique_json(Path(plan[\"r0_input_config\"]))\n        binding = make_native_binding(\n            plan, admission, config, project=str(ROOT / \"native_project\"))\n        require_native_binding_consensus(binding)\n        require(binding[\"identity\"] == identity and\n                binding[\"overrides\"][\"epochs\"] == 100 and\n                binding[\"overrides\"][\"batch\"] == 16 and\n                binding[\"overrides\"][\"resume\"] is False, \"FROZEN_RECIPE\")\n        verify_two_t4_runtime(torch.cuda)\n        native = load_pinned_native_class(binding)\n        require(native.__name__ == \"EarlyPairedTrainer\", \"NATIVE_ARM\")\n        trainer = native(**binding[\"constructor\"],\n                         overrides=binding[\"overrides\"])\n        require(trainer.world_size == 2 and trainer.ddp is False,\n                \"TWO_RANK_NATIVE\")\n        if rank == 0:\n            target = Path(trainer.best)\n            require(not target.exists(), \"NATIVE_BEST_ALREADY_EXISTS\")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(G4 / \"best.pt\", target)\n            require(sha256(target) == sha256(G4 / \"best.pt\"),\n                    \"BEST_COPY_SHA\")\n        dist.barrier()\n    finally:\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n    # Historical FRESH plan above authenticates the R0 input only.\n    # The actual optimizer/model/EMA/RNG state is always native RESUME.\n    trainer._ddp_b16_g4_identity = dict(identity)\n    trainer._ddp_b16_g4_root = str(G4.resolve(strict=True))\n    trainer._ddp_b16_native_state_request = {\n        \"schema\": STATE_SCHEMA, \"mode\": \"RESUME_FROM_DDP_B16\",\n        \"identity\": identity, \"cursor\": cursor,\n        \"checkpoint_root\": str(G4.resolve(strict=True)),\n        \"externally_pinned_manifest_sha256\": EXTERNAL_PIN,\n    }\n    trainer._ddp_b16_verified_worker_ready = True\n    trainer._ddp_b16_g4_native_bound = True\n    trainer._ddp_b16_terminal_candidate_ready = True\n\n    # Validate the REAL native post-G4 restore, before any training batch.\n    # The nominal launcher epoch alone is NOT sufficient evidence.\n    from research.r9_training.ddp_b16 import ddp_native_state as native_state\n    _real_native_start = native_state.ddp_b16_native_state_start\n    _expected_cursor = int(os.environ[\"DDP_B16_EXPECTED_NEXT_EPOCH_INDEX\"])\n    _expected_updates = int(os.environ[\"DDP_B16_EXPECTED_OPTIMIZER_UPDATES\"])\n\n    def _checked_native_state_start(bound_trainer):\n        last_step = _real_native_start(bound_trainer)\n        require(getattr(bound_trainer, \"_ddp_b16_native_restored\", False) is True,\n                \"NATIVE_G4_RESTORATION_NOT_CONFIRMED\")\n        require(bound_trainer.start_epoch == _expected_cursor,\n                \"REAL_NATIVE_RESTORE_WRONG_EPOCH\")\n        require(getattr(bound_trainer, \"_d4_optimizer_steps_observed\", None)\n                == _expected_updates, \"REAL_NATIVE_OPTIMIZER_CURSOR_DRIFT\")\n        require(dist.is_initialized() and dist.get_backend() == \"nccl\",\n                \"REAL_NATIVE_NCCL_BACKEND\")\n        if rank == 0:\n            print(\"STRICT_NATIVE_RESUME_ACCEPTED=\" +\n                  json.dumps({\"starting_epoch\": _expected_cursor + 1,\n                              \"completed_epochs\": _expected_cursor,\n                              \"optimizer_updates\": _expected_updates,\n                              \"manifest_sha256\": EXTERNAL_PIN},sort_keys=True),\n                  flush=True)\n        return last_step\n\n    native_state.ddp_b16_native_state_start = _checked_native_state_start\n\n\n    # Governed alternate numerical trajectory. Preserve all frozen guard hooks.\n    # Apply scale change only at a two-rank clean accumulation boundary.\n    state_amp = {\"applied\": False, \"branch_batch\": None,\n                 \"epoch\": None, \"batch\": -1, \"updates_fixed\": 0}\n    native_callbacks = trainer.run_callbacks\n    def amp128_callbacks(event):\n        if event == \"on_train_epoch_start\":\n            state_amp[\"epoch\"] = int(trainer.epoch)\n            state_amp[\"batch\"] = -1\n            if state_amp[\"epoch\"] > 95:\n                require(state_amp[\"applied\"] and\n                        float(trainer.scaler.get_scale()) == 128.0,\n                        \"AMP_POLICY_NEXT_EPOCH_SCALE\")\n        if event == \"on_train_batch_start\":\n            state_amp[\"batch\"] += 1\n            if state_amp[\"epoch\"] == 95 and not state_amp[\"applied\"]:\n                pending = sum(1 for p in trainer.model.parameters()\n                              if p.grad is not None)\n                consensus = [None, None]\n                dist.all_gather_object(consensus, {\n                    \"rank\": rank, \"batch\": state_amp[\"batch\"],\n                    \"pending_grad_tensors\": pending,\n                    \"scale\": float(trainer.scaler.get_scale()),\n                })\n                require([s[\"rank\"] for s in consensus] == [0, 1] and\n                        len({s[\"batch\"] for s in consensus}) == 1,\n                        \"AMP_POLICY_RANK_CURSOR_CONSENSUS\")\n                if all(s[\"pending_grad_tensors\"] == 0 for s in consensus):\n                    require(all(s[\"scale\"] == 1024.0 for s in consensus),\n                            \"AMP_POLICY_EXPECT_RESTORED_1024\")\n                    require(trainer.scaler.is_enabled(),\n                            \"AMP_POLICY_AMP_DISABLED\")\n                    scale_tensor = getattr(trainer.scaler, \"_scale\", None)\n                    require(isinstance(scale_tensor, torch.Tensor),\n                            \"AMP_POLICY_SCALE_TENSOR_MISSING\")\n                    scale_tensor.fill_(128.0)\n                    require(float(trainer.scaler.get_scale()) == 128.0,\n                            \"AMP_POLICY_CHANGE_FAILED\")\n                    native_update = trainer.scaler.update\n                    def fixed_scale_update(new_scale=None):\n                        require(new_scale is None,\n                                \"AMP_POLICY_EXTERNAL_UPDATE_OVERRIDE\")\n                        before = float(trainer.scaler.get_scale())\n                        require(before == 128.0,\n                                \"AMP_POLICY_PRE_UPDATE_DRIFT\")\n                        # This executes only after the native optimizer step.\n                        # Original D4 guards reject any skipped update first.\n                        result = native_update(new_scale=128.0)\n                        require(float(trainer.scaler.get_scale()) == 128.0,\n                                \"AMP_POLICY_POST_UPDATE_DRIFT\")\n                        state_amp[\"updates_fixed\"] += 1\n                        return result\n                    trainer.scaler.update = fixed_scale_update\n                    state_amp[\"applied\"] = True\n                    state_amp[\"branch_batch\"] = state_amp[\"batch\"]\n                    if rank == 0:\n                        print(\"AMP128_GOVERNED_BRANCH=\" +\n                              json.dumps({\n                                  \"epoch\": 96, \"zero_based_batch\": state_amp[\"batch\"],\n                                  \"original_loss_scale\": 1024,\n                                  \"fixed_loss_scale\": 128,\n                                  \"clean_accumulation_on_both_ranks\": True,\n                                  \"policy_sha256\": POLICY_SHA,\n                                  \"no_official_relabeling\": True,\n                              }, sort_keys=True), flush=True)\n                elif state_amp[\"batch\"] >= 16:\n                    raise RuntimeError(\n                        \"MV00_AMP128_NO_CLEAN_RANK_CONSENSUS_BOUNDARY\")\n            if state_amp[\"applied\"]:\n                require(float(trainer.scaler.get_scale()) == 128.0,\n                        \"AMP_POLICY_BATCH_SCALE_DRIFT\")\n        result = native_callbacks(event)\n        if event == \"on_train_batch_end\" and rank == 0 and (\n                state_amp[\"epoch\"] == 95 and state_amp[\"batch\"] == 696):\n            print(\"AMP128_PASSED_ORIGINAL_FAILURE_BATCH\", flush=True)\n        return result\n    trainer.run_callbacks = amp128_callbacks\n\n    # Faster crash reporting on a peer abort; no change to optimizer math.\n    original_init_process_group = dist.init_process_group\n    def bounded_nccl_init(*args, **kwargs):\n        if kwargs.get(\"backend\") == \"nccl\":\n            require(kwargs.get(\"timeout\") == timedelta(seconds=10800),\n                    \"UNEXPECTED_FROZEN_NCCL_TIMEOUT\")\n            kwargs[\"timeout\"] = timedelta(seconds=900)\n        return original_init_process_group(*args, **kwargs)\n    dist.init_process_group = bounded_nccl_init\n\n    start = time.monotonic()\n    last_epoch_end = start\n    state = {\"manifest\": EXTERNAL_PIN, \"next_epoch\": cursor[\"next_epoch\"]}\n    old_hook = hook.ddp_b16_g4_epoch_end\n\n    def session_hook(obj, *, epoch, nb, last_opt_step):\n        nonlocal last_epoch_end\n        require(state_amp[\"applied\"] and state_amp[\"branch_batch\"] == 1,\n                \"AMP_POLICY_UNVERIFIED_CLEAN_BOUNDARY\")\n        require(state_amp[\"epoch\"] == epoch and\n                state_amp[\"batch\"] == 889,\n                \"AMP_POLICY_EXPECT_FULL_EPOCH\")\n        require(float(obj.scaler.get_scale()) == 128.0,\n                \"AMP_POLICY_SCALE_AT_G4\")\n        synchronized = [None, None]\n        dist.all_gather_object(synchronized, {\n            \"rank\": rank, \"scale\": float(obj.scaler.get_scale()),\n            \"branch_batch\": state_amp[\"branch_batch\"],\n            \"updates_fixed\": state_amp[\"updates_fixed\"],\n            \"epoch\": epoch,\n        })\n        require([s[\"rank\"] for s in synchronized] == [0, 1] and\n                all(s[\"scale\"] == 128.0 and s[\"branch_batch\"] == 1 and\n                    s[\"epoch\"] == epoch and s[\"updates_fixed\"] > 0\n                    for s in synchronized),\n                \"AMP_POLICY_EPOCH_CONSENSUS\")\n        if rank == 0:\n            print(\"AMP128_EPOCH_FULL_TRAIN_VALIDATED=\" +\n                  json.dumps({\"completed_epoch\": epoch + 1,\n                              \"scale\": 128,\n                              \"policy_sha256\": POLICY_SHA,\n                              \"updates_fixed_per_rank\": [x[\"updates_fixed\"]\n                                                         for x in synchronized]},\n                             sort_keys=True), flush=True)\n        result = old_hook(obj, epoch=epoch, nb=nb, last_opt_step=last_opt_step)\n        if epoch == 99:\n            require(result.get(\"status\") ==\n                    \"TERMINAL_EPOCH_G4_CAPSULE_NOT_WRITTEN\",\n                    \"TERMINAL_STATUS\")\n            return result\n        require(result.get(\"status\") ==\n                \"LOCAL_SYNCHRONIZED_DDP_CAPSULE_EXTERNAL_AUTHENTICATION_PENDING\",\n                \"EPOCH_G4\")\n        require(epoch == state[\"next_epoch\"], \"EPOCH_SEQUENCE\")\n        consensus = [None, None]\n        dist.all_gather_object(consensus, {\n            \"rank\": rank,\n            \"manifest\": result[\"manifest_sha256\"],\n            \"best\": result[\"selected_best_sha256\"],\n        })\n        require([x[\"rank\"] for x in consensus] == [0, 1] and\n                len({x[\"manifest\"] for x in consensus}) == 1 and\n                len({x[\"best\"] for x in consensus}) == 1,\n                \"G4_RANK_CONSENSUS\")\n        message = [None]\n        if rank == 0:\n            try:\n                saved = persist_epoch(obj, result, identity, epoch,\n                                      state[\"manifest\"])\n                now = time.monotonic()\n                elapsed = now - start\n                this_epoch = now - last_epoch_end\n                stop = (elapsed >= BUDGET_HOURS * 3600 or\n                        elapsed + max(5400.0, 1.5 * this_epoch)\n                        >= 11.0 * 3600)\n                message[0] = {\n                    \"ok\": True, \"stop\": stop,\n                    \"manifest_sha256\": saved[\"manifest_sha256\"],\n                }\n            except Exception as exc:\n                message[0] = {\"ok\": False, \"error\": str(exc)[:300]}\n        dist.broadcast_object_list(message, src=0)\n        require(message[0][\"ok\"],\n                \"EXPORT_\" + str(message[0].get(\"error\", \"failed\")))\n        require(message[0][\"manifest_sha256\"] == result[\"manifest_sha256\"],\n                \"EXPORT_HASH_RANK\")\n        state[\"manifest\"] = result[\"manifest_sha256\"]\n        state[\"next_epoch\"] = epoch + 1\n        last_epoch_end = time.monotonic()\n        dist.barrier()\n        if message[0][\"stop\"]:\n            raise EpochBoundaryStop(\"SESSION_BUDGET_REACHED_AFTER_G4\")\n        return result\n\n    hook.ddp_b16_g4_epoch_end = session_hook\n    try:\n        safe = [np._core.multiarray.scalar, np.dtype]\n        safe.extend(type(np.dtype(n)) for n in (\n            \"float16\", \"float32\", \"float64\", \"int16\", \"int32\",\n            \"int64\", \"uint8\", \"uint16\", \"uint32\", \"uint64\", \"bool\",\n        ))\n        safe = list(dict.fromkeys(safe))\n        try:\n            with torch.serialization.safe_globals(safe):\n                trainer.train()\n            require(state[\"next_epoch\"] == 99 and trainer.epoch == 99,\n                    \"UNEXPECTED_NATIVE_STOP\")\n            status = \"EPOCH100_COMPLETED\"\n        except EpochBoundaryStop:\n            status = \"PAUSED_AT_G4\"\n\n        done = [None]\n        if rank == 0:\n            try:\n                terminal = persist_terminal(trainer, identity) if (\n                    status == \"EPOCH100_COMPLETED\") else None\n                done[0] = {\n                    \"ok\": True, \"status\": status,\n                    \"next_epoch\": 100 if terminal else state[\"next_epoch\"],\n                    \"manifest_sha256\": state[\"manifest\"],\n                    \"terminal\": terminal,\n                    \"training_executed\": True,\n                    \"b_test_access\": \"NONE\",\n                }\n            except Exception as exc:\n                done[0] = {\"ok\": False, \"error\": str(exc)[:300]}\n        dist.broadcast_object_list(done, src=0)\n        require(done[0][\"ok\"],\n                \"TERMINAL_\" + str(done[0].get(\"error\", \"unknown\")))\n        if rank == 0:\n            print(\"MV00_CONTINUOUS_RESULT=\" +\n                  json.dumps(done[0], sort_keys=True), flush=True)\n        dist.barrier()\n    finally:\n        dist.init_process_group = original_init_process_group\n        trainer.run_callbacks = native_callbacks\n        hook.ddp_b16_g4_epoch_end = old_hook\n        if dist.is_initialized():\n            dist.destroy_process_group()\n\n\nif __name__ == \"__main__\":\n    main()\n"

# MV00 DDP-B16 CONTINUOUS TRAINING — Kaggle Save Version / Save & Run All.
# Pin the previous output's manifest SHA and mounted dataset root on each new version.
# NO B-TEST. Fixed science / two T4 / batch16 / imgsz1024 / seed42 / 100 epochs.
# AUTO examines the attached Kaggle notebook outputs, but ONLY accepts
# the single saved G4 whose actual MANIFEST SHA-256 matches this exact pin.
PREVIOUS_OUTPUT_DATASET = "/kaggle/working/MV00_E95_INPUT_STAGED"
PREVIOUS_MANIFEST_SHA256 = "b31c817be3677ce7cecaf9fa2a6d7a177f2927d54dde5b055fd7ab8f69195745"
EXPECTED_COMPLETED_EPOCH = 95
EXPECTED_OPTIMIZER_UPDATES = None  # derived from externally SHA-pinned G4

from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys
import torch

SOURCE_HEAD = "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"
BUNDLE_SHA256 = "4dc26b7277d58ed0b1fbcc1f4fac68aaf7ad417951e735bc8be2b040764861bb"
BUNDLE = Path("/kaggle/input/datasets/awaishit/resema-ddp-b16-acceptance/RESEMA_DDP_B16_ACCEPTANCE.bundle")
WORK = Path("/kaggle/working/MV00_B16_CONTINUOUS")
REPO, STAGE, G4 = WORK/"repo", WORK/"stage", WORK/"g4"
OUT = Path("/kaggle/working/MV00_B16_CONTINUOUS_OUTPUT")
RUN_ID = "MV00_B16_FRESH_ACCEPTANCE"
INPUTS = {
 "n1":Path("/kaggle/input/datasets/awaishit/detection-frozen-git-n1-accepted-r1"),
 "handoff":Path("/kaggle/input/datasets/awaishit/detection-project-training-handoff-r1"),
 "legacy":Path("/kaggle/input/datasets/awaishit/detection-legacy-notebook-files-flat-r1"),
 "split_b":Path("/kaggle/input/datasets/utopianstar/grazpedwri-dx-split-b"),
 "assets":Path("/kaggle/input/datasets/nithoukhan/d4e-r12-accepted-assets"),
 "archives":Path("/kaggle/input/datasets/nithoukhan/d4e-r12-raw-archives-v1"),
 "evidence":Path("/kaggle/input/datasets/nithoukhan2/d4f-r7d-legacy-exact-source-evidence-r1"),
}
def sha256(p):
 h=hashlib.sha256()
 with Path(p).open("rb") as f:
  for b in iter(lambda:f.read(4*1024*1024),b""):h.update(b)
 return h.hexdigest()

# Use the exact approved pinned Git bundle; its Kaggle mount path can vary.
if not BUNDLE.is_file():
 print("HARD_CODED_SOURCE_BUNDLE_NOT_MOUNTED:", BUNDLE, flush=True)
 candidates = sorted(p for p in Path("/kaggle/input").rglob("*.bundle")
                     if p.is_file() and not p.is_symlink())
 verified = [p for p in candidates if sha256(p) == BUNDLE_SHA256]
 if not verified:
  raise FileNotFoundError(
   "PINNED_SOURCE_BUNDLE_NOT_ATTACHED. Expected SHA256="
   + BUNDLE_SHA256
   + ". Attach the dataset containing RESEMA_DDP_B16_ACCEPTANCE.bundle. "
   + "Discovered .bundle paths: " + str([str(p) for p in candidates[:30]])
  )
 BUNDLE = verified[0]
print("PINNED_GIT_BUNDLE_SELECTED:", BUNDLE, flush=True)
assert BUNDLE.is_file() and not BUNDLE.is_symlink()
assert all(p.is_dir() for p in INPUTS.values()), "MISSING_ORIGINAL_INPUT_DATASET"
assert not WORK.exists() and not OUT.exists(), "USE_CLEAN_KAGGLE_SESSION"
assert sha256(BUNDLE)==BUNDLE_SHA256, "SOURCE_BUNDLE_MISMATCH"
assert torch.cuda.is_available() and torch.cuda.device_count()==2
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1))

# Fail-closed source discovery: do not fall back to the epoch-1 dataset.
# Search ONLY attached Kaggle inputs, not the internet or old checkpoints.
# A matching mount path alone is not accepted: the off-session manifest SHA
# must match the independently recorded epoch-95 pin; verify bytes below.
requested_root = Path(PREVIOUS_OUTPUT_DATASET)
roots = []
if requested_root.is_dir():
 roots.append(requested_root)
else:
 print("SELECTED_NOTEBOOK_OUTPUT_MISSING_AT:", requested_root, flush=True)
 # Kaggle sometimes mounts an attached Notebook Output under a different
 # owner/slug or allows saving it as a Dataset with a different mount root.
 # Scope fallback to notebook inputs and checkpoint-named dataset inputs.
 notebooks_root = Path("/kaggle/input/notebooks")
 if notebooks_root.is_dir():
  roots.append(notebooks_root)
 datasets_root = Path("/kaggle/input/datasets")
 if datasets_root.is_dir():
  for owner in sorted(datasets_root.iterdir()):
   if not owner.is_dir() or owner.is_symlink():
    continue
   for dataset in sorted(owner.iterdir()):
    if (dataset.is_dir() and not dataset.is_symlink() and
        any(term in dataset.name.lower()
            for term in ("mv00", "ddp-b16", "continuous-output", "grazpedwri"))):
     roots.append(dataset)
 top = Path("/kaggle/input")
 if top.is_dir():
  for candidate in sorted(top.iterdir()):
   if (candidate.is_dir() and not candidate.is_symlink()
       and any(term in candidate.name.lower()
               for term in ("mv00", "ddp-b16", "continuous-output", "grazpedwri"))):
    roots.append(candidate)
roots = list(dict.fromkeys(roots))
print("CHECKPOINT_SEARCH_ROOTS:", [str(p) for p in roots], flush=True)

accepted, visible = [], []
for root in roots:
 for candidate in sorted(root.rglob("ACCEPTANCE.json")):
  if not candidate.is_file() or candidate.is_symlink() or candidate.parent.name != "g4":
   continue
  try:
   metadata = json.loads(candidate.read_text("utf-8"))
  except (OSError, ValueError):
   continue
  digest = metadata.get("manifest_sha256")
  visible.append((str(candidate), digest, metadata.get("completed_epoch")))
  if (digest == PREVIOUS_MANIFEST_SHA256 and
      metadata.get("run_id") == RUN_ID and
      metadata.get("source_release_sha") == SOURCE_HEAD and
      metadata.get("completed_epoch") == EXPECTED_COMPLETED_EPOCH and
      type(metadata.get("optimizer_updates")) is int and
      metadata["optimizer_updates"] > 14647 and
      metadata.get("schema") == "MV00_DDP_B16_CONTINUOUS_OUTPUT_V1" and
      metadata.get("status") == "MV00_RESUMABLE_EPOCH_G4_CANDIDATE" and
      metadata.get("b_test_access") == "NONE"):
   accepted.append(candidate)

accepted = list(dict.fromkeys(accepted))
if len(accepted) != 1:
 print("VISIBLE_G4_RECEIPTS:", visible, flush=True)
 print("ATTACHED_INPUT_TOP_LEVEL:",
       [str(x) for x in sorted(Path("/kaggle/input").iterdir())]
       if Path("/kaggle/input").is_dir() else [], flush=True)
 raise AssertionError(
  "MV00_EPOCH95_INPUT_NOT_ATTACHED_OR_WRONG_VERSION: "
  "Attach the Kaggle Notebook Output containing the exact epoch-95 "
  "ACCEPTANCE.json and G4 manifest SHA-256 "
  + PREVIOUS_MANIFEST_SHA256
  + ". Do NOT use epoch-95, MV01, or a similarly named other output."
 )
PRIOR = accepted[0].parent.parent
print("PINNED_CHECKPOINT_SELECTED:", accepted[0],flush=True)
print("PINNED_CHECKPOINT_SHA256:", PREVIOUS_MANIFEST_SHA256,flush=True)

receipt=json.loads(accepted[0].read_text("utf-8"))
src=accepted[0].parent
assert receipt["schema"] == "MV00_DDP_B16_CONTINUOUS_OUTPUT_V1", "MV00_CONTINUOUS_RECEIPT_REQUIRED"
assert receipt["status"] == "MV00_RESUMABLE_EPOCH_G4_CANDIDATE", "G4_CANDIDATE_REQUIRED"
assert type(receipt["optimizer_updates"]) is int and receipt["optimizer_updates"] > 14647, "SAVED_OPTIMIZER_CURSOR_INVALID"
# Adopt the count ONLY from the receipt whose manifest is checked against the
# independent SHA pin. Confirm it also matches that manifest before execution.
EXPECTED_OPTIMIZER_UPDATES = receipt["optimizer_updates"]
assert receipt["completed_epoch"] == EXPECTED_COMPLETED_EPOCH, (
 "WRONG_START_CHECKPOINT_COMPLETED_EPOCH"
)
assert receipt["next_epoch"] == EXPECTED_COMPLETED_EPOCH, (
 "WRONG_START_CHECKPOINT_NEXT_EPOCH"
)
assert receipt["optimizer_updates"] == EXPECTED_OPTIMIZER_UPDATES, (
 "WRONG_START_CHECKPOINT_OPTIMIZER_UPDATES"
)
assert receipt["manifest_sha256"]==PREVIOUS_MANIFEST_SHA256, "EXTERNAL_CHECKPOINT_PIN_DRIFT"
assert receipt["run_id"]==RUN_ID and receipt["source_release_sha"]==SOURCE_HEAD
assert receipt["b_test_access"]=="NONE"
assert receipt["status"] in {
 "NONOFFICIAL_EPOCH1_G4_SAVED_NOT_OFFSESSION_VERIFIED",
 "MV00_RESUMABLE_EPOCH_G4_CANDIDATE"
}, "G4_RECEIPT_STATUS"
files=receipt["files_sha256"]
assert len(files)==6
assert all(Path(name).name==name and (src/name).is_file() and
           not (src/name).is_symlink() and sha256(src/name)==digest
           for name,digest in files.items()),"G4_MEMBER_SHA_FAILURE"
manifests=[name for name in files if name.endswith("_two_rank_manifest.json")]
assert len(manifests)==1
assert sha256(src/manifests[0])==PREVIOUS_MANIFEST_SHA256
manifest=json.loads((src/manifests[0]).read_text("utf-8"))
assert manifest["complete"] is True and set(manifest["ranks"])=={"0","1"}
assert manifest["identity"]["run_id"]==RUN_ID
assert manifest["identity"]["arm"]=="MV00"
assert manifest["identity"]["ddp_release_sha"]==SOURCE_HEAD
assert 1<=manifest["cursor"]["next_epoch"]<100
assert manifest["cursor"]["next_epoch"] == EXPECTED_COMPLETED_EPOCH, (
 "MANIFEST_CURSOR_WRONG_RESUME_EPOCH"
)
assert manifest["cursor"]["completed_epoch"] == EXPECTED_COMPLETED_EPOCH - 1, (
 "MANIFEST_LAST_COMPLETED_INDEX_MISMATCH"
)
assert manifest["cursor"]["optimizer_updates"] == EXPECTED_OPTIMIZER_UPDATES, (
 "MANIFEST_OPTIMIZER_UPDATES_MISMATCH"
)
assert manifest["b_test_access"] == "NONE"
assert manifest["schema"] == "VGRA_DDP_B16_G4_EPOCH_PAIR_CANDIDATE_V1"
assert manifest["world_size"] == 2 and manifest["complete"] is True
assert manifest["cursor"]["sampler_next_epoch"] == EXPECTED_COMPLETED_EPOCH
assert manifest["cursor"]["steps_per_rank"] == 890
assert manifest["cursor"]["global_batch"] == 16
assert files["best.pt"] == receipt["selected_best_sha256"]
for rank in (0, 1):
 member = manifest["ranks"][str(rank)]
 assert files[member["filename"]] == member["sha256"], "RANK_PT_SHA"
 assert files[member["receipt_filename"]] == member["receipt_sha256"], "RANK_RECEIPT_SHA"
 rank_receipt = json.loads((src/member["receipt_filename"]).read_text("utf-8"))
 assert rank_receipt["rank"] == rank
 assert rank_receipt["identity"] == manifest["identity"]
 assert rank_receipt["cursor"] == manifest["cursor"]
 assert rank_receipt["sha256"] == member["sha256"]
print("MV00 EPOCH-95 OFF-SESSION G4 AUTHENTICATION: PASS", flush=True)
print("PINNED OPTIMIZER UPDATES:", EXPECTED_OPTIMIZER_UPDATES, flush=True)
print("STRICT_PRE_TRAINING_RESUME_PASS: completed",EXPECTED_COMPLETED_EPOCH,
      "starting",EXPECTED_COMPLETED_EPOCH+1,
      "optimizer_updates",EXPECTED_OPTIMIZER_UPDATES,flush=True)

WORK.mkdir()
subprocess.run(["git","clone","--quiet","--branch","research/vgra-ddp-b16",
                str(BUNDLE),str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",SOURCE_HEAD],
               check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                               text=True).strip()==SOURCE_HEAD
assert not subprocess.check_output(["git","-C",str(REPO),"status","--porcelain",
                                    "--untracked-files=no"],text=True).strip()
G4.mkdir()
for name,digest in files.items():
 shutil.copyfile(src/name,G4/name)
 assert sha256(G4/name)==digest

sys.dont_write_bytecode=True
sys.path.insert(0,str(REPO))
from research.r9_training.ddp_b16.r0_relocation import restore_r0_config
from research.r9_training.ddp_b16.ddp_worker import review_worker_inputs
from research.r9_training.ddp_b16.ddp_verified_admission import verify_staged_admission

config_files=sorted(INPUTS["n1"].rglob("R0_RELOCATED_CONFIG_CANDIDATE.json"))
assert len(config_files)==1
config=config_files[0]
r0_file=REPO/"research/r9_training/frozen_source/ultralytics/models/yolo/detect/d4e_r3_read_only_identity_preflight.py"
spec=importlib.util.spec_from_file_location("_ddp_b16_continuous_r0",r0_file)
r0=importlib.util.module_from_spec(spec)
spec.loader.exec_module(r0)
restore_r0_config(config,[INPUTS[k] for k in ("n1","handoff","legacy","assets","archives","evidence")],
 asset_pins=r0.ASSET_PINS,overlay_pins=r0.OVERLAY_PINS,code_pins=r0.R21_PINS,
 native_blob_pin=r0.NATIVE_BLOB_SHA1,
 runtime_yaml_pin="f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",
 mirrored_handoff=INPUTS["handoff"],
 archives_roots=[INPUTS["assets"],INPUTS["archives"]])
plan={
 "schema":"VGRA_DDP_B16_WORKER_PLAN_REVIEW_V1","arm":"MV00",
 "run_id":RUN_ID,"mode":"FRESH_FROM_EARLY", # source review only; actual state RESUME
 "checkout_root":str(REPO),"expected_git_head":SOURCE_HEAD,
 "r0_input_config":str(config),"stage_root":str(STAGE),"b_test_access":"NONE"
}
plan_path=WORK/"plan.json"
plan_path.write_text(json.dumps(plan,sort_keys=True,indent=2)+"\n","utf-8")
review=review_worker_inputs(plan,stage_source=True)
admission=verify_staged_admission(plan,review)
assert admission["identity"]==manifest["identity"],"G4_SOURCE_ADMISSION_DRIFT"

OUT.mkdir()
POLICY_ID="MV00_AMP128_CAPPED_FAILCLOSED_RECOVERY_V1"
POLICY_RECORD={
 "schema":"MV00_DDP_B16_AMP_NUMERICAL_RECOVERY_POLICY_V1",
 "policy_id":POLICY_ID,
 "arm":"MV00",
 "run_id":RUN_ID,
 "original_source_release_sha":SOURCE_HEAD,
 "starting_manifest_sha256":PREVIOUS_MANIFEST_SHA256,
 "starting_completed_epochs":EXPECTED_COMPLETED_EPOCH,
 "starting_optimizer_updates":EXPECTED_OPTIMIZER_UPDATES,
 "original_restored_amp_scale":1024,
 "amp_scale":128,
 "scale_mode":"FIXED_AT_128_AFTER_TWO_RANK_CLEAN_BOUNDARY",
 "numerical_safety_guards":"UNMODIFIED_D4_FAIL_CLOSED",
 "model_architecture_changed":False,
 "dataset_split_changed":False,
 "scientific_original_trajectory":False,
 "official_training_authorized":False,
 "b_test_access":"NONE",
 "r3_evidence":"BATCH696_PASSED_BOTH_RANKS_SCALE128_DIAGNOSTIC",
}
policy_path=OUT/"AMP128_RECOVERY_POLICY.json"
policy_path.write_text(json.dumps(POLICY_RECORD,indent=2,sort_keys=True)+"\n",
                       encoding="utf-8")
POLICY_SHA256=sha256(policy_path)
print("AMP128_RECOVERY_POLICY_SHA256:",POLICY_SHA256,flush=True)
worker_file=WORK/"multiepoch_worker.py"
worker_file.write_text(WORKER_CODE,encoding="utf-8")
env=dict(os.environ)
env.update({"CUDA_VISIBLE_DEVICES":"0,1","OMP_NUM_THREADS":"1",
            "PYTHONDONTWRITEBYTECODE":"1","PYTHONPATH":str(REPO)+os.pathsep+env.get("PYTHONPATH",""),
            "DDP_B16_PREVIOUS_MANIFEST_SHA256":PREVIOUS_MANIFEST_SHA256,
            "DDP_B16_SESSION_HOURS":"9.0",
            "DDP_B16_EXPECTED_NEXT_EPOCH_INDEX":str(EXPECTED_COMPLETED_EPOCH),
            "DDP_B16_EXPECTED_OPTIMIZER_UPDATES":str(EXPECTED_OPTIMIZER_UPDATES),
            "DDP_B16_AMP128_POLICY_SHA256":POLICY_SHA256,
            "TORCH_DISTRIBUTED_DEBUG":"DETAIL",
            "TORCH_NCCL_DESYNC_DEBUG":"1",
            "TORCH_FR_BUFFER_SIZE":"2048",
            "TORCH_NCCL_DUMP_ON_TIMEOUT":"1"})
cmd=[sys.executable,"-m","torch.distributed.run","--standalone","--nnodes=1",
     "--nproc_per_node=2","--max_restarts=0",str(worker_file)]
print("MV00 B16 CONTINUOUS RUN START; RESUMING EPOCH",
      manifest["cursor"]["next_epoch"]+1,"OF 100",flush=True)
print("SOURCE",SOURCE_HEAD,"AMP128 MODIFIED NUMERICAL TRAJECTORY; B-TEST SEALED",flush=True)
log=OUT/"MV00_CONTINUOUS_LOG.txt"
with log.open("x",encoding="utf-8") as file:
 process=subprocess.run(cmd,cwd=REPO,env=env,stdout=file,stderr=subprocess.STDOUT)
tail=log.read_text("utf-8")
assert tail.count("STRICT_NATIVE_RESUME_ACCEPTED=") == 1, (
 "NATIVE_TRAINER_DID_NOT_CONFIRM_EXPECTED_EPOCH_AND_OPTIMIZER_CURSOR; "
 + str(log)
)
results=[s.split("=",1)[1] for s in tail.splitlines()
         if s.startswith("MV00_CONTINUOUS_RESULT=")]
if process.returncode or len(results)!=1:
 print("AMP128_RECOVERY_FAILED. NO NEW SCIENTIFIC RESULT ACCEPTED.")
 latest_receipt=OUT/"g4"/"ACCEPTANCE.json"
 if latest_receipt.is_file():
  last_good=json.loads(latest_receipt.read_text("utf-8"))
  print("LAST EXPORTED AMP128 G4:",last_good.get("completed_epoch"),
        last_good.get("manifest_sha256"),flush=True)
 print(tail[-12000:])
 raise RuntimeError("AMP128_CONTINUATION_FAILED_INSPECT_"+str(log))
result=json.loads(results[0])
assert result["b_test_access"]=="NONE" and result["training_executed"] is True
assert tail.count("AMP128_GOVERNED_BRANCH=")==1,"AMP128_BRANCH_MISSING_OR_DUPLICATED"
assert "AMP128_PASSED_ORIGINAL_FAILURE_BATCH" in tail,"BATCH696_NOT_PASSED"
if result["status"]=="PAUSED_AT_G4":
 latest=OUT/"g4"/"ACCEPTANCE.json"
 assert latest.is_file()
 new=json.loads(latest.read_text("utf-8"))
 assert new["manifest_sha256"]==result["manifest_sha256"]
 assert new["amp128_recovery_branch"] is True
 assert new["amp128_policy_id"]==POLICY_ID
 assert new["amp128_policy_sha256"]==POLICY_SHA256
 assert new["amp_scale_at_checkpoint"]==128.0
 assert sha256(OUT/"g4"/"AMP128_RECOVERY_POLICY.json")==POLICY_SHA256
 assert new["next_epoch"]>manifest["cursor"]["next_epoch"]
 for name,digest in new["files_sha256"].items():
  assert sha256(OUT/"g4"/name)==digest
 print("SESSION PASS. NEXT EPOCH:",new["next_epoch"]+1,"/100")
 print("PERSISTED MANIFEST SHA:",new["manifest_sha256"])
 print("SAVE VERSION AND ATTACH ITS OUTPUT TO THE NEXT SESSION.")
else:
 assert result["status"]=="EPOCH100_COMPLETED"
 terminal_file=OUT/"terminal"/"TERMINAL.json"
 assert terminal_file.is_file()
 terminal=json.loads(terminal_file.read_text("utf-8"))
 assert terminal["completed_epochs"]==100
 assert terminal["amp128_recovery_branch"] is True
 assert terminal["amp128_policy_id"]==POLICY_ID
 assert terminal["amp128_policy_sha256"]==POLICY_SHA256
 assert terminal["amp_scale_at_terminal"]==128.0
 assert terminal["operational_val_images"]==3049
 assert sha256(OUT/"terminal"/"AMP128_RECOVERY_POLICY.json")==POLICY_SHA256
 for name,digest in terminal["files_sha256"].items():
  assert sha256(OUT/"terminal"/name)==digest,"TERMINAL_FILE_HASH"
 print("AMP128 EPOCH100 COMPLETED. AMENDED TRAJECTORY; TERMINAL B-VAL CANDIDATE.")
print("OUTPUT:",OUT,"B-TEST SEALED")
